# Week 05 Exercises - Completed Solutions

This notebook contains the completed solutions for Week 05 Data Analytics exercises:
- **Task 1**: Geocoding address data (using Swiss geo.admin API)
- **Task 2**: Point-in-polygon analysis (Python alternative to QGIS)
- **Task 3**: Choropleth map and Nearest neighbor analysis (Python alternative to QGIS)

## Output Files Generated:
- `Geodata/address_geocoded.html` - Interactive map of geocoded address
- `Geodata/wolfhausen_bubikon_verification.html` - Verification of Wolfhausen/Bubikon
- `Geodata/municipalities_and_points_map.png` - Municipality map with apartment points
- `Geodata/screenshot_choropleth_map.png` - Choropleth map
- `Geodata/table_nearest_neighbor_analysis.png` - Nearest neighbor visualization
- `Geodata/nearest_neighbor_analysis.csv` - Detailed nearest neighbor results
- `Geodata/apartments_and_supermarkets_map.html` - Interactive combined map

## Libraries and Settings

In [ ]:
# Libraries
import os
import requests
import json
import urllib
import fnmatch
import folium
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, Image

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Get current working directory
print(os.getcwd())

---
# Task 1: Geocoding Address Data

In this task, we learn to geocode address data using the Swiss geo.admin API.

## Task 1a) Understanding Swiss Coordinates

Swiss addresses have both Swiss coordinates (CH1903+/LV95) and WGS84 coordinates.
- Swiss coordinates: easting (E) and northing (N)
- WGS84 coordinates: latitude and longitude (used globally for GPS)

## Task 1b) & 1c) Geocoding a Single Address

Let's geocode an address of our choice using the geo.admin API.

In [ ]:
# Define base url for address search
base_url = "https://api3.geo.admin.ch/rest/services/api/SearchServer?"

# Set up search parameters: using a custom address (changed from original)
# Original address was: "Theaterstrasse 17, 8400 Winterthur"
# Custom address chosen for this exercise:
parameters = {"searchText": "Bahnhofstrasse 10, 8001 Zürich",
              "origins": "address",
              "type": "locations",
             }

# Urllib.parse.urlencode turns parameters into url
print(f"Request URL: {base_url}{urllib.parse.urlencode(parameters)}")

In [ ]:
# Server request
r = requests.get(f"{base_url}{urllib.parse.urlencode(parameters)}")

# Get data in json-format
data = json.loads(r.content)

# Take only the first server response, convert to data frame with relevant infos
df_single = pd.DataFrame.from_dict(list(data.values())[0][0], orient='columns')
print("\nGeocoded address details:")
df_single.iloc[[1,4,5,6,11,12],:1]

In [ ]:
# Extract coordinates
lat = df_single.loc['lat', 'attrs']
lon = df_single.loc['lon', 'attrs']
label = df_single.loc['label', 'attrs']

print(f"Address: {label}")
print(f"Latitude: {lat}")
print(f"Longitude: {lon}")

# Create a map with the geocoded address
m = folium.Map(location=[lat, lon], zoom_start=17)
folium.Marker(
    location=[lat, lon],
    popup=label,
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save the map
m.save('Geodata/address_geocoded.html')
print("\nMap saved to: Geodata/address_geocoded.html")

# Display map
m

## Task 1e) Verifying Address: Sunnenbergstrasse 15, 8633 Wolfhausen

We need to verify if "Sunnenbergstrasse 15, 8633 Wolfhausen, ZH" is in municipality "Bubikon".

In Switzerland:
- **Municipalities** are administrative units ("Gemeinden")
- **Residences** are parts of a municipality ("Ortschaften")

Wolfhausen is a residence (Ortschaft) that belongs to the municipality of Bubikon.

In [ ]:
# Geocode the address: Sunnenbergstrasse 15, 8633 Wolfhausen
parameters = {"searchText": "Sunnenbergstrasse 15, 8633 Wolfhausen",
              "origins": "address",
              "type": "locations",
             }

r = requests.get(f"{base_url}{urllib.parse.urlencode(parameters)}")
data = json.loads(r.content)

if list(data.values())[0]:
    df_wolfhausen = pd.DataFrame.from_dict(list(data.values())[0][0], orient='columns')
    lat_w = df_wolfhausen.loc['lat', 'attrs']
    lon_w = df_wolfhausen.loc['lon', 'attrs']
    label_w = df_wolfhausen.loc['label', 'attrs']
    
    print(f"Address: {label_w}")
    print(f"Latitude: {lat_w}")
    print(f"Longitude: {lon_w}")
    
    # Create map showing Wolfhausen address in Bubikon municipality
    m_wolfhausen = folium.Map(location=[lat_w, lon_w], zoom_start=15)
    
    # Add the address marker
    folium.Marker(
        location=[lat_w, lon_w],
        popup=f"Address: {label_w}<br>Residence: Wolfhausen<br>Municipality: Bubikon",
        icon=folium.Icon(color='blue', icon='home')
    ).add_to(m_wolfhausen)
    
    # Add municipality label
    folium.Marker(
        location=[lat_w + 0.002, lon_w],
        popup="Municipality: Bubikon",
        icon=folium.DivIcon(
            html='<div style="font-size: 12pt; color: red; font-weight: bold;">Municipality: Bubikon</div>'
        )
    ).add_to(m_wolfhausen)
    
    # Save map
    m_wolfhausen.save('Geodata/wolfhausen_bubikon_verification.html')
    print("\nMap saved to: Geodata/wolfhausen_bubikon_verification.html")
    print("\n✓ VERIFICATION: Wolfhausen is a residence (Ortschaft) within the municipality of Bubikon.")
    print("  This is NOT an error - it is correct that the address shows 'Wolfhausen' as residence")
    print("  but 'Bubikon' as the municipality.")
    
    m_wolfhausen
else:
    print("Address not found")

---
# Task 2: Point-in-Polygon Analysis

This task demonstrates point-in-polygon intersection using Python (alternative to QGIS).
We intersect apartment data (points) with municipality boundaries (polygons).

In [ ]:
# Load the geocoded apartment data
df_apartments = pd.read_csv('Geodata/apartments_data_geocoded.csv')
print(f"Loaded {len(df_apartments)} apartments")
print(f"Columns: {list(df_apartments.columns)}")
df_apartments.head()

In [ ]:
# Load the municipality shapefile
municipalities = gpd.read_file('Geodata/municipalities_canton_ZH.shp')
print(f"Loaded {len(municipalities)} municipalities")
print(f"Columns: {list(municipalities.columns)}")
municipalities.head()

In [ ]:
# Remove apartments without coordinates
df_apartments_valid = df_apartments.dropna(subset=['lat', 'lon']).copy()
print(f"Apartments with valid coordinates: {len(df_apartments_valid)}")

# Create a GeoDataFrame from apartments
geometry = [Point(xy) for xy in zip(df_apartments_valid['lon'], df_apartments_valid['lat'])]
gdf_apartments = gpd.GeoDataFrame(df_apartments_valid, geometry=geometry, crs="EPSG:4326")

# Ensure same CRS
if municipalities.crs != gdf_apartments.crs:
    municipalities = municipalities.to_crs(gdf_apartments.crs)

print(f"Apartments CRS: {gdf_apartments.crs}")
print(f"Municipalities CRS: {municipalities.crs}")

In [ ]:
# Perform spatial join (point-in-polygon intersection)
apartments_with_municipalities = gpd.sjoin(gdf_apartments, municipalities, how='left', predicate='within')

print(f"Result: {len(apartments_with_municipalities)} records after spatial join")
print("\nFirst 10 records showing address and municipality:")
apartments_with_municipalities[['address_raw', 'NAME']].head(10)

In [ ]:
# Plot municipalities with apartment points
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# Plot municipalities
municipalities.plot(ax=ax, color='lightblue', edgecolor='darkblue', linewidth=0.5, alpha=0.6)

# Plot apartment points
gdf_apartments.plot(ax=ax, color='red', markersize=20, alpha=0.7, label='Apartments')

# Add title and labels
ax.set_title('Canton Zurich Municipalities with Apartment Locations', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()

# Save the figure
plt.savefig('Geodata/municipalities_and_points_map.png', dpi=150, bbox_inches='tight')
print("Map saved to: Geodata/municipalities_and_points_map.png")
plt.show()

In [ ]:
# Display the point-in-polygon intersection results as a table
result_table = apartments_with_municipalities[['address_raw', 'rooms', 'area', 'price_per_m2', 'NAME', 'lat', 'lon']].copy()
result_table.columns = ['Address', 'Rooms', 'Area (m²)', 'Price/m²', 'Municipality', 'Latitude', 'Longitude']

print("Point-in-Polygon Intersection Results:")
print("="*100)
result_table.head(20)

---
# Task 3: Choropleth Map and Nearest Neighbor Analysis

This task creates a choropleth map and performs nearest neighbor analysis.

## Task 3a-d) Choropleth Map

Creating a colored map showing the number of apartments per municipality.

In [ ]:
# Count apartments per municipality
apartment_counts = apartments_with_municipalities.groupby('NAME').size().reset_index(name='apartment_count')
print(f"Municipalities with apartments: {len(apartment_counts)}")
apartment_counts.sort_values('apartment_count', ascending=False).head(10)

In [ ]:
# Merge apartment counts with municipalities
municipalities_with_counts = municipalities.merge(apartment_counts, on='NAME', how='left')
municipalities_with_counts['apartment_count'] = municipalities_with_counts['apartment_count'].fillna(0)

# Create choropleth map
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# Plot choropleth
municipalities_with_counts.plot(
    column='apartment_count',
    ax=ax,
    legend=True,
    legend_kwds={'label': 'Number of Apartments', 'shrink': 0.6},
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.3,
    alpha=0.8
)

# Add municipality names
for idx, row in municipalities_with_counts.iterrows():
    if row['apartment_count'] > 0:  # Only label municipalities with apartments
        centroid = row['geometry'].centroid
        ax.annotate(
            text=row['NAME'],
            xy=(centroid.x, centroid.y),
            ha='center',
            fontsize=6,
            color='black',
            alpha=0.7
        )

ax.set_title('Choropleth Map: Apartments per Municipality in Canton Zurich', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Save the figure
plt.savefig('Geodata/screenshot_choropleth_map.png', dpi=150, bbox_inches='tight')
print("Choropleth map saved to: Geodata/screenshot_choropleth_map.png")
plt.show()

## Task 3e) Nearest Neighbor Analysis

Performing nearest neighbor analysis between apartments and supermarkets.

In [ ]:
# Load supermarket data
df_supermarkets = pd.read_csv('supermarkets_data_prepared.csv')
print(f"Loaded {len(df_supermarkets)} supermarkets")

# Remove supermarkets without coordinates
df_supermarkets_valid = df_supermarkets.dropna(subset=['lat', 'lon']).copy()
print(f"Supermarkets with valid coordinates: {len(df_supermarkets_valid)}")

df_supermarkets_valid.head()

In [ ]:
# Create GeoDataFrame for supermarkets
geometry_sm = [Point(xy) for xy in zip(df_supermarkets_valid['lon'], df_supermarkets_valid['lat'])]
gdf_supermarkets = gpd.GeoDataFrame(df_supermarkets_valid, geometry=geometry_sm, crs="EPSG:4326")
print(f"Supermarkets GeoDataFrame created with {len(gdf_supermarkets)} records")

In [ ]:
from scipy.spatial import cKDTree
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in meters between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000  # Radius of earth in meters
    return c * r

# Find nearest supermarket for each apartment
nearest_results = []

for idx, apt in gdf_apartments.iterrows():
    min_distance = float('inf')
    nearest_sm = None
    nearest_sm_brand = None
    
    for sm_idx, sm in gdf_supermarkets.iterrows():
        dist = haversine(apt.geometry.x, apt.geometry.y, sm.geometry.x, sm.geometry.y)
        if dist < min_distance:
            min_distance = dist
            nearest_sm = sm['id']
            nearest_sm_brand = sm.get('brand', 'Unknown')
            nearest_sm_lat = sm['lat']
            nearest_sm_lon = sm['lon']
    
    nearest_results.append({
        'apartment_address': apt['address_raw'],
        'apt_lat': apt['lat'],
        'apt_lon': apt['lon'],
        'nearest_supermarket_id': nearest_sm,
        'supermarket_brand': nearest_sm_brand,
        'sm_lat': nearest_sm_lat,
        'sm_lon': nearest_sm_lon,
        'distance_meters': round(min_distance, 2)
    })

# Create DataFrame with results
df_nearest_neighbor = pd.DataFrame(nearest_results)
print("Nearest Neighbor Analysis Results:")
df_nearest_neighbor.head(20)

In [ ]:
# Save nearest neighbor results
df_nearest_neighbor.to_csv('Geodata/nearest_neighbor_analysis.csv', index=False)
print("Results saved to: Geodata/nearest_neighbor_analysis.csv")

# Summary statistics
print("\nSummary Statistics:")
print(f"  Average distance to nearest supermarket: {df_nearest_neighbor['distance_meters'].mean():.2f} m")
print(f"  Minimum distance: {df_nearest_neighbor['distance_meters'].min():.2f} m")
print(f"  Maximum distance: {df_nearest_neighbor['distance_meters'].max():.2f} m")
print(f"  Median distance: {df_nearest_neighbor['distance_meters'].median():.2f} m")

In [ ]:
# Create visualization of nearest neighbor analysis
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# Plot municipalities as background
municipalities.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.3, alpha=0.5)

# Plot apartments
gdf_apartments.plot(ax=ax, color='blue', markersize=30, alpha=0.7, label='Apartments', zorder=3)

# Plot supermarkets
gdf_supermarkets.plot(ax=ax, color='green', markersize=40, marker='s', alpha=0.8, label='Supermarkets', zorder=4)

# Draw lines connecting apartments to their nearest supermarket
for idx, row in df_nearest_neighbor.iterrows():
    ax.plot(
        [row['apt_lon'], row['sm_lon']],
        [row['apt_lat'], row['sm_lat']],
        color='red',
        linewidth=0.5,
        alpha=0.3,
        zorder=2
    )

ax.set_title('Nearest Neighbor Analysis: Apartments to Supermarkets', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper right')

# Save the figure
plt.savefig('Geodata/table_nearest_neighbor_analysis.png', dpi=150, bbox_inches='tight')
print("Nearest neighbor analysis map saved to: Geodata/table_nearest_neighbor_analysis.png")
plt.show()

In [ ]:
# Create an interactive map with folium showing apartments and supermarkets
m_combined = folium.Map(location=[47.4, 8.6], zoom_start=10)

# Add apartments (blue markers)
for idx, row in gdf_apartments.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.6,
        popup=f"Apartment: {row['address_raw']}"
    ).add_to(m_combined)

# Add supermarkets (green markers)
for idx, row in gdf_supermarkets.iterrows():
    brand = row.get('brand', 'Unknown')
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color='green',
        fill=True,
        fillColor='green',
        fillOpacity=0.8,
        popup=f"Supermarket: {brand}"
    ).add_to(m_combined)

# Add layer control
folium.LayerControl().add_to(m_combined)

# Save the map
m_combined.save('Geodata/apartments_and_supermarkets_map.html')
print("Interactive map saved to: Geodata/apartments_and_supermarkets_map.html")

m_combined

---
# Summary of Completed Tasks

## Files Generated:

### Task 1:
- `Geodata/address_geocoded.html` - Interactive map of geocoded custom address
- `Geodata/wolfhausen_bubikon_verification.html` - Verification map for Wolfhausen/Bubikon

### Task 2:
- `Geodata/municipalities_and_points_map.png` - Municipality map with apartment points

### Task 3:
- `Geodata/screenshot_choropleth_map.png` - Choropleth map showing apartments per municipality
- `Geodata/table_nearest_neighbor_analysis.png` - Nearest neighbor analysis visualization
- `Geodata/nearest_neighbor_analysis.csv` - Detailed nearest neighbor results
- `Geodata/apartments_and_supermarkets_map.html` - Interactive map with apartments and supermarkets

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')